# [1.6] Local Frontier ML Infrastructure - Solutions

**Core Question.** How do you keep frontier interpretability notebooks honest on one local GPU?

This solved notebook executes the same contract as the learner notebook and interprets the committed CUDA report.

## Learning Objectives

- Verify environment, memory, parity, generation, and activation-store checks.
- Distinguish estimates from measured CUDA allocations.
- Keep the claim boundary narrow.

> ```yaml
> Difficulty: 3
> Importance: 5
> ```

<details>
<summary>Help - how to read this solved notebook</summary>

Passing infrastructure tests make later work auditable; they do not replace model-specific oracles.

</details>

In [1]:
GT_TIER = "GT-1"
EXERCISE_ID = "1_6_local_frontier_ml_infrastructure"
DIFFICULTY = 3
IMPORTANCE = 5
EXPECTED_RUNTIME = "seconds for toy contracts; minutes for the local CUDA runtime report"
REQUIRES_GPU = True

import json
import sys
import tempfile
from pathlib import Path

import torch as t

chapter = "chapter1_transformer_interp"
section = "part6_frontier_ml_infrastructure"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section

if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part6_frontier_ml_infrastructure.tests as tests
import part6_frontier_ml_infrastructure.utils as utils

from arena_ext import (
    DiskActivationStore,
    compare_logits,
    deterministic_generation_equal,
    estimate_inference_memory,
    get_environment_report,
)

MAIN = True

from chapter1_transformer_interp.exercises.part6_frontier_ml_infrastructure import solutions

## Exercise 1 - Environment Checks

<details>
<summary>Expected output</summary>

The report should include Python, PyTorch, CUDA, GPU name, and BF16 support.

</details>

In [2]:
environment = solutions.run_environment_check(required_vram_gb=24.0)
environment.as_dict()

Environment report
  python: 3.14.6
  platform: Linux-7.1.2-3-cachyos-x86_64-with-glibc2.43
  torch: 2.12.1+cu132
  cuda_available: True
  cuda_version: 13.2
  gpu_name: NVIDIA GeForce RTX 5090 Laptop GPU
  gpu_total_memory_gb: 23.45965576171875
  bf16_supported: True
  flash_attn_available: False
  xformers_available: False


{'python': '3.14.6',
 'platform': 'Linux-7.1.2-3-cachyos-x86_64-with-glibc2.43',
 'torch': '2.12.1+cu132',
 'cuda_available': True,
 'cuda_version': '13.2',
 'gpu_name': 'NVIDIA GeForce RTX 5090 Laptop GPU',
 'gpu_total_memory_gb': 23.45965576171875,
 'bf16_supported': True,
 'flash_attn_available': False,
 'xformers_available': False}

## Exercise 2 - VRAM Budget Estimates

In [3]:
budget = solutions.estimate_gemma_1b_smoke_budget()
utils.print_dict_table("Estimated memory", budget.as_dict())
tests.test_memory_budget_fits_local_tier(solutions.estimate_gemma_1b_smoke_budget)
tests.test_memory_budget_rejects_oversized_model()

Estimated memory
  parameter_gb  : 1.862645149230957
  kv_cache_gb   : 0.28125
  activation_gb : 0.015625
  optimizer_gb  : 0.0
  overhead_gb   : 1.5
  total_gb      : 3.659520149230957
All tests in `test_memory_budget_fits_local_tier` passed!
All tests in `test_memory_budget_rejects_oversized_model` passed!


## Exercise 3 - HF Parity and Drift Controls

<details>
<summary>Interpreting parity</summary>

The positive case should pass tight tolerances; the drift fixture should fail them.

</details>

In [4]:
tests.test_compare_logits_detects_match(solutions.compare_logits)
tests.test_compare_logits_rejects_shape_mismatch(solutions.compare_logits)
tests.test_compare_logits_rejects_real_drift(solutions.compare_logits)
tests.test_hf_parity_smoke_test_passes(solutions.hf_parity_smoke_test)

All tests in `test_compare_logits_detects_match` passed!
All tests in `test_compare_logits_rejects_shape_mismatch` passed!
All tests in `test_compare_logits_rejects_real_drift` passed!
ParityReport(max_abs_diff=3.2633543014526367e-06, mse=1.0065202794493078e-12, kl_divergence=-8.670995033099871e-10, topk_agreement=1.0)
All tests in `test_hf_parity_smoke_test_passes` passed!


## Exercise 4 - Generation Parity

In [5]:
tests.test_deterministic_generation_equal_detects_mismatch(
    solutions.deterministic_generation_equal,
)
assert solutions.generation_parity_smoke_test()

All tests in `test_deterministic_generation_equal_detects_mismatch` passed!


## Exercise 5 - Activation Storage

In [6]:
with tempfile.TemporaryDirectory() as tmpdir:
    tests.test_disk_activation_store_roundtrip(Path(tmpdir))
    tests.test_activation_store_smoke_test_contract(
        Path(tmpdir),
        solutions.activation_store_smoke_test,
    )

All tests in `test_disk_activation_store_roundtrip` passed!
All tests in `test_activation_store_smoke_test_contract` passed!


## Exercise 6 - Notebook Contract

In [7]:
contract = solutions.run_smoke_test(cpu=True)
assert contract["hf_parity_passed"]
assert contract["generation_parity_passed"]
assert contract["activation_store"]["num_records"] == 2
tests.test_notebook_contract(solutions.run_smoke_test)
contract

Environment report
  python: 3.14.6
  platform: Linux-7.1.2-3-cachyos-x86_64-with-glibc2.43
  torch: 2.12.1+cu132
  cuda_available: True
  cuda_version: 13.2
  gpu_name: NVIDIA GeForce RTX 5090 Laptop GPU
  gpu_total_memory_gb: 23.45965576171875
  bf16_supported: True
  flash_attn_available: False
  xformers_available: False
ParityReport(max_abs_diff=3.2633543014526367e-06, mse=1.0065202794493078e-12, kl_divergence=-8.670995033099871e-10, topk_agreement=1.0)
Environment report
  python: 3.14.6
  platform: Linux-7.1.2-3-cachyos-x86_64-with-glibc2.43
  torch: 2.12.1+cu132
  cuda_available: True
  cuda_version: 13.2
  gpu_name: NVIDIA GeForce RTX 5090 Laptop GPU
  gpu_total_memory_gb: 23.45965576171875
  bf16_supported: True
  flash_attn_available: False
  xformers_available: False
ParityReport(max_abs_diff=3.2633543014526367e-06, mse=1.0065202794493078e-12, kl_divergence=-8.670995033099871e-10, topk_agreement=1.0)
All tests in `test_notebook_contract` passed!


{'environment': {'python': '3.14.6',
  'platform': 'Linux-7.1.2-3-cachyos-x86_64-with-glibc2.43',
  'torch': '2.12.1+cu132',
  'cuda_available': True,
  'cuda_version': '13.2',
  'gpu_name': 'NVIDIA GeForce RTX 5090 Laptop GPU',
  'gpu_total_memory_gb': 23.45965576171875,
  'bf16_supported': True,
  'flash_attn_available': False,
  'xformers_available': False},
 'budget': {'parameter_gb': 1.862645149230957,
  'kv_cache_gb': 0.017578125,
  'activation_gb': 0.0009765625,
  'optimizer_gb': 0.0,
  'overhead_gb': 1.5,
  'total_gb': 3.381199836730957},
 'hf_parity_passed': True,
 'generation_parity_passed': True,
 'activation_store': {'root': '/tmp/tmp90umku1i',
  'num_records': 2,
  'total_values': 28,
  'names': ['resid_pre', 'mlp_out']}}

## Signature Result

| Check | Observed | Required |
|---|---:|---:|
| Python / PyTorch | `3.14.6` / `2.12.1+cu132` | `3.14` / `2.12.1+cu132` |
| CUDA runtime | `13.2` | `13.2` |
| BF16 CUDA matmul | `[1024,1024]`, finite | finite BF16 tensor |
| `uv pip check` | pass | pass |
| Peak VRAM | `0.043 GB` | `< 1 GB` |

## Limitations

This does not load a checkpoint or benchmark a model. It proves the local evidence harness is ready.

## Bonus - Anomaly Hunting

- Raise the estimate above 24GB and confirm the budget check fails.
- Change logit ordering and confirm top-k parity fails.
- Delete activation metadata and inspect what the report can no longer prove.

In [8]:
report = json.loads((section_dir / "verification_report.json").read_text())
gpu = report["metrics"]["gpu_test"]
assert report["accepted"]
assert gpu["cuda_available"]
assert gpu["gpu_tensor_test_passed"]
assert gpu["gpu_matmul_dtype"] == "torch.bfloat16"
assert gpu["peak_vram_gb"] < 1.0
print("python_major_minor:", gpu["python_major_minor"])
print("torch_version:", gpu["torch_version"])
print("cuda_version:", gpu["cuda_version"])
print("peak_vram_gb:", gpu["peak_vram_gb"])

tests.test_committed_gpu_report_records_cuda_runtime_and_no_fallback()
tests.test_exercise_notebook_course_ready_surface()

python_major_minor: 3.14
torch_version: 2.12.1+cu132
cuda_version: 13.2
peak_vram_gb: 0.04296875
All tests in `test_committed_gpu_report_records_cuda_runtime_and_no_fallback` passed!
All tests in `test_exercise_notebook_course_ready_surface` passed!
